In [0]:
%run ../../02_common_utils/operations

In [0]:
from datetime import datetime

team_name  = "team_lemma"
bronze_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"

try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {staging_db}.finwire_parsed LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "Batch1"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "Batch1"
    
run_id=carried_run_id
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit
from pyspark.sql.types import DecimalType, LongType

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'staging_to_silver_market_finwire', f'Starting processing for staging to silver FINWIRE parsing (batch: {carried_batch})')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'RUNNING')


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number


cmp = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType") == "CMP")

w_cmp = Window.partitionBy("CompanyID").orderBy(col("EffectiveDate").desc())

silver_company = (
    cmp
    .withColumn("rn", row_number().over(w_cmp))
    .filter(col("rn") == 1)
    .drop("rn")
    .select(
        col("CompanyID").alias("companyid"),
        trim(col("CompanyName")).alias("companyname"),
        trim(col("Status")).alias("status"),
        trim(col("IndustryID")).alias("industryid"),
        trim(col("SPrating")).alias("sprating"),
        col("FoundingDate").alias("foundingdate"),
        trim(col("AddrLine1")).alias("addrline1"),
        trim(col("AddrLine2")).alias("addrline2"),
        trim(col("PostalCode")).alias("postalcode"),
        trim(col("City")).alias("city"),
        trim(col("StateProvince")).alias("stateprovince"),
        trim(col("Country")).alias("country"),
        trim(col("CEOname")).alias("ceoname"),
        trim(col("Description")).alias("description"),
        col("EffectiveDate").alias("effectivedate"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
    )
)

(silver_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_db}.company"))

count = spark.table(f"{silver_db}.company").count()
print(f"silver.company: {count:,}")


In [0]:
from pyspark.sql.functions import col, trim, row_number, current_timestamp, lit, when

sec = spark.table(f"{staging_db}.finwire_parsed").filter(
    col("RecType").isin("SEC_CIK", "SEC_NAME")
)


company_lookup = (
    spark.table(f"{silver_db}.company")
    .select(
        col("companyid").alias("lkp_companyid"),
        col("companyname").alias("lkp_name")
    )
)

sec_resolved = (
    sec
    .join(company_lookup,
          trim(col("CoNameOrCIK")) == trim(col("lkp_name")),
          how="left")
    .withColumn("resolved_companyid",
        when(col("RecType") == "SEC_CIK",
             trim(col("CoNameOrCIK")).cast(LongType()))
        .otherwise(col("lkp_companyid"))   # SEC_NAME → from join
    )
    .drop("lkp_companyid", "lkp_name")
)


w_sec = Window.partitionBy("Symbol").orderBy(col("EffectiveDate").desc())

silver_security = (
    sec_resolved
    .withColumn("rn", row_number().over(w_sec))
    .filter(col("rn") == 1)
    .drop("rn")
    .select(
        trim(col("Symbol")).alias("symbol"),
        trim(col("IssueType")).alias("issuetype"),
        trim(col("Status")).alias("status"),
        trim(col("Name")).alias("name"),
        trim(col("ExID")).alias("exchangeid"),
        col("ShOut").alias("sharesoutstanding"),
        col("FirstTradeDate").alias("firsttradedate"),
        col("FirstTradeExchg").alias("firsttradeonexchange"),
        col("Dividend").alias("dividend"),
        trim(col("CoNameOrCIK")).alias("conameorcik"),
        col("resolved_companyid").alias("companyid"),
        col("EffectiveDate").alias("effectivedate"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
    )
)

(silver_security.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_db}.security"))

count = spark.table(f"{silver_db}.security").count()
print(f"silver.security: {count:,}")


In [0]:

fin = spark.table(f"{staging_db}.finwire_parsed").filter(
    col("RecType").isin("FIN_COMPANYID", "FIN_NAME")
)


fin_resolved = (
    fin
    .join(company_lookup,
          trim(col("CoNameOrCIK")) == trim(col("lkp_name")),
          how="left")
    .withColumn("resolved_companyid",
        when(col("RecType") == "FIN_COMPANYID",
             trim(col("CoNameOrCIK")).cast(LongType()))
        .otherwise(col("lkp_companyid"))
    )
    .drop("lkp_companyid", "lkp_name")
)


silver_financial = (
    fin_resolved
    .select(
        col("resolved_companyid").alias("companyid"),
        col("FIYear").alias("fi_year"),
        col("FIQtr").alias("fi_qtr"),
        col("QtrStartDate").alias("fi_qtr_start_date"),
        col("PostingDate").alias("fi_posting_date"),
        col("Revenue").alias("fi_revenue"),
        col("Earnings").alias("fi_net_earn"),
        col("EPS").alias("fi_basic_eps"),
        col("DilutedEPS").alias("fi_dilut_eps"),
        col("Margin").alias("fi_margin"),
        col("Inventory").alias("fi_inventory"),
        col("Assets").alias("fi_assets"),
        col("Liabilities").alias("fi_liability"),
        col("ShOut").alias("fi_out_basic"),
        col("DilutedShOut").alias("fi_out_dilut"),
        trim(col("CoNameOrCIK")).alias("conameorcik"),
        col("EffectiveDate").alias("effectivedate"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
    )
)

(silver_financial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_db}.financial"))

count = spark.table(f"{silver_db}.financial").count()
print(f"silver.financial: {count:,}")


In [0]:
checks = {
    
    "silver.company"       : (spark.table(f"{silver_db}.company").count(),         4_596),
    "silver.security"      : (spark.table(f"{silver_db}.security").count(),        7_598),
    "silver.financial"     : (spark.table(f"{silver_db}.financial").count(),     457_025),
}

print(f"\n{'Table':<25} {'Actual':>10} {'Expected':>10} {'Status'}")
print("-" * 55)
for tbl, (actual, expected) in checks.items():
    status = "PASS" if actual == expected else " FAIL"
    print(f"{tbl:<25} {actual:>10,} {expected:>10,}  {status}")

##AUDIT LOG

In [0]:
%run ../../02_common_utils/operations

In [0]:

source_count = (
    spark.table(f"{staging_db}.finwire_parsed")
    .filter(col("RecType") == "CMP")
    .count()
)
target_count = spark.table(f"{silver_db}.company").count()
carried_run_id = str(
    spark.table(f"{silver_db}.company").select("`_run_id`").first()[0]
)

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="Batch1",
    domain="MARKET",
    table_name="company",
    source_layer="staging",
    target_layer="silver",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="Batch1",
    layer="silver",
    table_name="company",
    operation="OVERWRITE",
    rows_affected=target_count
)

print(f"source_count : {source_count:,}")   
print(f"target_count : {target_count:,}")   


In [0]:

source_count = (
    spark.table(f"{staging_db}.finwire_parsed")
    .filter(col("RecType").isin("SEC_CIK", "SEC_NAME"))
    .count()
)
target_count = spark.table(f"{silver_db}.security").count()
carried_run_id = str(
    spark.table(f"{silver_db}.security").select("`_run_id`").first()[0]
)

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="Batch1",
    domain="MARKET",
    table_name="security",
    source_layer="staging",
    target_layer="silver",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="Batch1",
    layer="silver",
    table_name="security",
    operation="OVERWRITE",
    rows_affected=target_count
)

print(f"source_count : {source_count:,}")  
print(f"target_count : {target_count:,}")   
                                           

In [0]:

source_count = (
    spark.table(f"{staging_db}.finwire_parsed")
    .filter(col("RecType").isin("FIN_COMPANYID", "FIN_NAME"))
    .count()
)
target_count = spark.table(f"{silver_db}.financial").count()
carried_run_id = str(
    spark.table(f"{silver_db}.financial").select("`_run_id`").first()[0]
)

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="Batch1",
    domain="MARKET",
    table_name="financial",
    source_layer="staging",
    target_layer="silver",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="Batch1",
    layer="silver",
    table_name="financial",
    operation="OVERWRITE",
    rows_affected=target_count
)


log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'staging_to_silver_market_finwire', 'Successfully completed processing for silver FINWIRE tables.')

print(f"source_count : {source_count:,}")   
print(f"target_count : {target_count:,}")   

In [0]:
# --- Company Audit ---
source_count_cmp = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType") == "CMP").count()
log_pipeline_recon(spark, carried_run_id, carried_batch, "MARKET", "company", "staging", "silver", source_count_cmp, count_company)
log_audit_event(spark, carried_run_id, carried_batch, "silver", "company", "OVERWRITE", count_company)

# --- Security Audit ---
source_count_sec = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType").isin("SEC_CIK", "SEC_NAME")).count()
log_pipeline_recon(spark, carried_run_id, carried_batch, "MARKET", "security", "staging", "silver", source_count_sec, count_security)
log_audit_event(spark, carried_run_id, carried_batch, "silver", "security", "OVERWRITE", count_security)

# --- Financial Audit ---
source_count_fin = spark.table(f"{staging_db}.finwire_parsed").filter(col("RecType").isin("FIN_COMPANYID", "FIN_NAME")).count()
log_pipeline_recon(spark, carried_run_id, carried_batch, "MARKET", "financial", "staging", "silver", source_count_fin, count_financial)
log_audit_event(spark, carried_run_id, carried_batch, "silver", "financial", "OVERWRITE", count_financial)

# ─── OPERATIONS: SUCCESS LOGGING ─────────────────────────────────────────
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'staging_to_silver_market_finwire', 'Successfully completed processing for silver FINWIRE tables.')